# ComplyPilot JB v2 - Advanced RAG Setup
이 노트북은 **Parent-Child Retriever**와 **BM25 Hybrid Search** 기법을 적용하여 법령 데이터베이스를 구축합니다.
- 부모 문서(Parent): 1500자 (전체 문맥 보존용, 로컬 파일 시스템 저장)
- 자식 문서(Child): 300자 (정밀 검색용 임베딩, Chroma DB 저장)
- 키워드 검색(BM25): 정확한 단어 매칭용 (Pickle로 저장)


In [1]:
import os
import glob
import shutil
import pickle
from dotenv import load_dotenv

# LangChain Imports
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.storage import LocalFileStore
from langchain_classic.retrievers import ParentDocumentRetriever, EnsembleRetriever

load_dotenv(dotenv_path='../../.env')
if not os.getenv('OPENAI_API_KEY'):
    print('Warning: OPENAI_API_KEY is not set.')
else:
    print('OpenAI API Key loaded.')


C:\Users\USER\AppData\Local\Temp\ipykernel_1620\3075845119.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
c:\Users\USER\Desktop\complypilot-jb\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OpenAI API Key loaded.


In [2]:
# 1. 기존 DB 및 저장소 초기화 (덮어쓰기 방지)
db_dirs = ['../chroma_db_child', '../parent_store']
for d in db_dirs:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"기존 {d} 폴더를 삭제했습니다.")

if os.path.exists('../bm25_retriever.pkl'):
    os.remove('../bm25_retriever.pkl')
    print("기존 bm25_retriever.pkl 삭제 완료")


기존 ../chroma_db_child 폴더를 삭제했습니다.
기존 ../parent_store 폴더를 삭제했습니다.


In [3]:
# 2. PDF 문서 로드
pdf_folder = '../data/vectordb'
pdf_files = glob.glob(os.path.join(pdf_folder, '*.pdf'))

all_documents = []
for file_path in pdf_files:
    try:
        loader = PyMuPDFLoader(file_path)
        docs = loader.load()
        for doc in docs:
            doc.metadata['source_type'] = 'regulation'
        all_documents.extend(docs)
    except Exception as e:
        print(f"로드 실패 ({os.path.basename(file_path)}): {e}")

print(f"총 로드된 페이지 수: {len(all_documents)}")


총 로드된 페이지 수: 232


In [4]:
# 3. Parent-Child 스플리터 및 스토어 설정
# - 부모(Parent): 1500자로 잘라서 전체 문맥을 통째로 유지합니다.
# - 자식(Child): 300자로 촘촘하게 잘라 벡터 검색의 정확도를 높입니다.
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

# 자식 문서를 담을 Vector DB (Chroma)
embedding_model = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma(
    collection_name="split_parents",
    embedding_function=embedding_model,
    persist_directory="../chroma_db_child"
)

# 부모 문서를 영구 보관할 로컬 스토리지
from langchain_classic.storage._lc_store import create_kv_docstore
fs = LocalFileStore("../parent_store")
store = create_kv_docstore(fs)

# Parent-Child 리트리버 생성
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)


In [5]:
# 4. 문서 추가 (Embedding 진행)
print("문서를 분석하고 임베딩을 시작합니다... (이 과정은 시간이 조금 걸릴 수 있습니다)")

# 문서를 넣으면 자동으로 부모/자식 분할 후 각각의 스토어에 저장됩니다.
parent_retriever.add_documents(all_documents, ids=None)
print("Parent-Child Retriever DB 구축 완료!")


문서를 분석하고 임베딩을 시작합니다... (이 과정은 시간이 조금 걸릴 수 있습니다)
Parent-Child Retriever DB 구축 완료!


In [6]:
# 5. BM25 키워드 검색기 구축 및 저장
print("BM25 키워드 검색용 모델을 구축합니다...")

# BM25에는 자식 문장이 아닌 '부모 문서(1500자)' 자체를 쪼개서 넣어야 합니다.
parent_docs = parent_splitter.split_documents(all_documents)
bm25_retriever = BM25Retriever.from_documents(parent_docs)

# 로컬 파일로 저장 (추후 02_main_workflow.ipynb 에서 불러오기 위함)
with open('../bm25_docs.pkl', 'wb') as f:
    pickle.dump(parent_docs, f)
print("BM25용 Document 피클 저장 완료! (../bm25_docs.pkl)")


BM25 키워드 검색용 모델을 구축합니다...
BM25용 Document 피클 저장 완료! (../bm25_docs.pkl)


In [7]:
# 6. 하이브리드 검색(Ensemble) 테스트
# Vector Search(70%) + Keyword Search(30%) 비율로 검색합니다.
ensemble_retriever = EnsembleRetriever(
    retrievers=[parent_retriever, bm25_retriever],
    weights=[0.7, 0.3]
)

query = "금융상품 광고 시 최고 이자율이나 수수료에 대해 어떻게 고지해야 하는지 알려줘"
results = ensemble_retriever.invoke(query)

print(f"질문: {query}")
print("\n[검색된 관련 근거 (부모 문맥 1500자 통째로 반환)]")
for i, res in enumerate(results[:2]): # 상위 2개만 출력
    print(f"\n--- 근거 {i+1} (출처: {res.metadata.get('source', '알 수 없음')}) ---")
    print(res.page_content)


질문: 금융상품 광고 시 최고 이자율이나 수수료에 대해 어떻게 고지해야 하는지 알려줘

[검색된 관련 근거 (부모 문맥 1500자 통째로 반환)]

--- 근거 1 (출처: ../data/vectordb\금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf) ---
법제처                                                            18                                                   국가법령정보센터
금융소비자 보호에 관한 감독규정
③ 금융상품판매업자등이 영 제18조제4항에 따라 일부 내용을 제외할 경우 준수해야 할 기준은 다음 각 호의 구
분에 따른다.
1. 보장성 상품에 관한 광고
가. 다음의 사항 전부 또는 일부만을 개괄적으로 알릴 것
　  1) 금융상품의 편익
　  2) 금융상품에 적합한 금융소비자의 특성 또는 가입요건
　  3) 금융상품의 특성
　  4) 판매채널의 특징 및 상담 연락처
나. 영상 또는 음성을 활용하는 광고인 경우에는 광고 시간이 2분 이내일 것
2. 그 밖의 금융상품에 관한 광고: 광고에 영 제18조제3항 각 호의 내용 중 일부를 제외함으로 인해 금융소비자의
합리적 의사결정이 저해되거나 건전한 시장질서가 훼손될 우려가 없을 것
 
제18조(광고의 방법 및 절차) 영 제19조제1항에서 "금융위원회가 정하여 고시하는 기준"이란 광고에서 글자의 색깔
ㆍ크기 또는 음성의 속도ㆍ크기 등이 해당 금융상품으로 인해 금융소비자가 받을 수 있는 혜택과 불이익을 균형
있게 전달할 것을 말한다.
 
제19조(광고 시 금지행위) ① 영 제20조제1항제6호에서 "금융위원회가 정하여 고시하는 행위"란 다음 각 호의 구분
에 따른 행위를 말한다.
1. 금융소비자에 따라 달라질 수 있는 거래조건을 누구에게나 적용될 수 있는 것처럼 오인하게 만드는 행위
2. 보험금 지급사유나 지급시점이 다름에도 불구하고 각각의 보험금이 한꺼번에 지급되는 것

In [8]:
query = "대출성 금융상품 광고에서 누구나 승인, 빠른 승인, 최저금리, 중도상환수수료 무료 표현을 사용할 때 필요한 조건 고지와 소비자 오인 방지 기준"

results = ensemble_retriever.invoke(query)

print(f"질문: {query}")
print("\n[검색된 관련 근거 (부모 문맥 1500자 통째로 반환)]")
for i, res in enumerate(results[:2]): # 상위 2개만 출력
    print(f"\n--- 근거 {i+1} (출처: {res.metadata.get('source', '알 수 없음')}) ---")
    print(res.page_content)

질문: 대출성 금융상품 광고에서 누구나 승인, 빠른 승인, 최저금리, 중도상환수수료 무료 표현을 사용할 때 필요한 조건 고지와 소비자 오인 방지 기준

[검색된 관련 근거 (부모 문맥 1500자 통째로 반환)]

--- 근거 1 (출처: ../data/vectordb\금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf) ---
법제처                                                            18                                                   국가법령정보센터
금융소비자 보호에 관한 감독규정
③ 금융상품판매업자등이 영 제18조제4항에 따라 일부 내용을 제외할 경우 준수해야 할 기준은 다음 각 호의 구
분에 따른다.
1. 보장성 상품에 관한 광고
가. 다음의 사항 전부 또는 일부만을 개괄적으로 알릴 것
　  1) 금융상품의 편익
　  2) 금융상품에 적합한 금융소비자의 특성 또는 가입요건
　  3) 금융상품의 특성
　  4) 판매채널의 특징 및 상담 연락처
나. 영상 또는 음성을 활용하는 광고인 경우에는 광고 시간이 2분 이내일 것
2. 그 밖의 금융상품에 관한 광고: 광고에 영 제18조제3항 각 호의 내용 중 일부를 제외함으로 인해 금융소비자의
합리적 의사결정이 저해되거나 건전한 시장질서가 훼손될 우려가 없을 것
 
제18조(광고의 방법 및 절차) 영 제19조제1항에서 "금융위원회가 정하여 고시하는 기준"이란 광고에서 글자의 색깔
ㆍ크기 또는 음성의 속도ㆍ크기 등이 해당 금융상품으로 인해 금융소비자가 받을 수 있는 혜택과 불이익을 균형
있게 전달할 것을 말한다.
 
제19조(광고 시 금지행위) ① 영 제20조제1항제6호에서 "금융위원회가 정하여 고시하는 행위"란 다음 각 호의 구분
에 따른 행위를 말한다.
1. 금융소비자에 따라 달라질 수 있는 거래조건을 누구에게나 적용될 수 있는 것처럼 오인하게 만드는 행위
2. 보험금 지급사유나 

In [9]:
query = "신용카드 광고에서 최대 캐시백, 전월 실적 없음, 연회비 무료 또는 부담 없음 표현을 사용할 때 혜택 조건과 제한사항을 어떻게 고지해야 하는지"

results = ensemble_retriever.invoke(query)

print(f"질문: {query}")
print("\n[검색된 관련 근거 (부모 문맥 1500자 통째로 반환)]")
for i, res in enumerate(results[:2]): # 상위 2개만 출력
    print(f"\n--- 근거 {i+1} (출처: {res.metadata.get('source', '알 수 없음')}) ---")
    print(res.page_content)

질문: 신용카드 광고에서 최대 캐시백, 전월 실적 없음, 연회비 무료 또는 부담 없음 표현을 사용할 때 혜택 조건과 제한사항을 어떻게 고지해야 하는지

[검색된 관련 근거 (부모 문맥 1500자 통째로 반환)]

--- 근거 1 (출처: ../data/vectordb\여신전문금융업감독규정(금융위원회고시)(제2026-17호)(20260506).pdf) ---
법제처                                                            22                                                   국가법령정보센터
여신전문금융업감독규정
절차와 기준을 정하여야 한다.
② 제1항에 따른 절차와 기준에는 다음 각 호의 사항이 포함되어야 한다.
1. 법 제14조의2제2항, 법 제14조의5제2항 및 제3항에서 정하는 준수사항 및 그 밖의 모집질서 유지를 위해 신용
카드업자와 소속 모집인이 준수하여야 할 사항
2. 제1호에 따른 준수사항의 이행여부 점검을 위한 전담 점검부서, 점검 방법, 정기적 점검 의무
3. 제1호의 준수사항 위반이 발생한 경우 재발 방지를 위한 대책 수립 및 이행
4. 제1호를 위반한 모집행위에 대해 인터넷 홈페이지 및 우편 등을 통한 신고제도 운영
 
제24조의14(결제대행업체의 준수사항) 시행령 제6조의16에 따라 결제대행업체는 신용카드회원등이 정기적으로 금
액을 지불하는 방식으로 물품이나 용역(「공공기관의 운영에 관한 법률」에 따른 공공기관이 제공하는 물품이나
용역,「전기통신사업법」에 따른 전기통신사업자가 제공하는 전기통신역무, 그 밖에 이와 유사한 것으로 금융위원
회가 정하여 고시하는 경우는 제외한다)을 구입하기 위해 신용카드등으로 거래(이하 ‘정기결제’ 라 한다)하는 경
우 다음 각 호의 방법 및 절차에 따른다.
1. 정기결제 방식으로 구입하는 물품이나 용역에 대해 결제 승인 요청 예정인 금액이 증가하거나 유료전환 되는
경우, 이와 관련한 사항을 정기결제 승인요청 7일 전까지 신

In [10]:
query = "예금 광고에서 원금 100% 안전, 고수익, 예금자보호 전액 보장 표현을 사용할 때 예금자보호 범위와 소비자 오인 방지 기준"
results = ensemble_retriever.invoke(query)

print(f"질문: {query}")
print("\n[검색된 관련 근거 (부모 문맥 1500자 통째로 반환)]")
for i, res in enumerate(results[:2]): # 상위 2개만 출력
    print(f"\n--- 근거 {i+1} (출처: {res.metadata.get('source', '알 수 없음')}) ---")
    print(res.page_content)


질문: 예금 광고에서 원금 100% 안전, 고수익, 예금자보호 전액 보장 표현을 사용할 때 예금자보호 범위와 소비자 오인 방지 기준

[검색된 관련 근거 (부모 문맥 1500자 통째로 반환)]

--- 근거 1 (출처: ../data/vectordb\금융소비자 보호에 관한 법률(법률)(제21065호)(20260102).pdf) ---
법제처                                                            11                                                       국가법령정보센터
금융소비자 보호에 관한 법률
라. 대출성 상품의 경우: 대출조건
4. 그 밖에 금융소비자 보호를 위하여 대통령령으로 정하는 내용
④ 금융상품판매업자등이 금융상품등에 관한 광고를 하는 경우 다음 각 호의 구분에 따른 행위를 해서는 아니 된다.
1. 보장성 상품
가. 보장한도, 보장 제한 조건, 면책사항 또는 감액지급 사항 등을 빠뜨리거나 충분히 고지하지 아니하여 제한 없
이 보장을 받을 수 있는 것으로 오인하게 하는 행위
나. 보험금이 큰 특정 내용만을 강조하거나 고액 보장 사례 등을 소개하여 보장내용이 큰 것으로 오인하게 하는
행위
다. 보험료를 일(日) 단위로 표시하거나 보험료의 산출기준을 불충분하게 설명하는 등 보험료등이 저렴한 것으로
오인하게 하는 행위
라. 만기 시 자동갱신되는 보장성 상품의 경우 갱신 시 보험료등이 인상될 수 있음을 금융소비자가 인지할 수 있
도록 충분히 고지하지 아니하는 행위
마. 금리 및 투자실적에 따라 만기환급금이 변동될 수 있는 보장성 상품의 경우 만기환급금이 보장성 상품의 만기
일에 확정적으로 지급되는 것으로 오인하게 하는 행위 등 금융소비자 보호를 위하여 대통령령으로 정하는 행
위
2. 투자성 상품
가. 손실보전(損失補塡) 또는 이익보장이 되는 것으로 오인하게 하는 행위. 다만, 금융소비자를 오인하게 할 우려
가 없는 경우로서 대통령령으로 정하는 경우는 제외한다.
나.